# Ch.5 — Semantic Segmentation (FCN, U-Net, DeepLab)

> **Mission:** Build pixel-level classification for retail shelf monitoring (ProductionCV challenge).  
> **Dataset:** Synthetic retail shelf images (background, products, empty space, shelf edges).  
> **Goal:** Achieve IoU ≥ 62% using U-Net architecture.

## 1 · The Core Idea: Classify Every Pixel

**Semantic segmentation** extends image classification from a single label to a label per pixel.

- **Image classification**: Input $224 \times 224 \times 3$ → Output $1000$ (class probabilities)
- **Semantic segmentation**: Input $H \times W \times 3$ → Output $H \times W \times C$ (per-pixel class probabilities)

**Three milestone architectures:**
1. **FCN (Fully Convolutional Network)**: Replace FC layers with 1×1 convs, upsample to input size
2. **U-Net**: Symmetric encoder-decoder with skip connections at every resolution level
3. **DeepLabV3+**: Atrous (dilated) convolutions + ASPP (multi-scale context)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Dark theme for plots (matches authoring guide)
plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#1a1a2e'

## 2 · Dataset: Synthetic Retail Shelf Images

Create synthetic retail shelf data with 5 classes:
- **0**: Background
- **1**: Products (cereal boxes, milk cartons)
- **2**: Empty shelf space
- **3**: Price tags
- **4**: Shelf edges

In [ ]:
class SyntheticShelfDataset(Dataset):
    """Generate synthetic retail shelf images with segmentation masks"""
    def __init__(self, num_samples=500, img_size=512):
        self.num_samples = num_samples
        self.img_size = img_size
        
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # Create synthetic shelf image
        image = np.zeros((self.img_size, self.img_size, 3), dtype=np.float32)
        mask = np.zeros((self.img_size, self.img_size), dtype=np.int64)
        
        # Background (class 0) - white walls
        image[:, :] = [0.9, 0.9, 0.9]
        
        # Shelf edges (class 4) - horizontal lines
        shelf_rows = [100, 200, 300, 400]
        for row in shelf_rows:
            image[row-5:row+5, :] = [0.3, 0.2, 0.1]  # Brown wood color
            mask[row-5:row+5, :] = 4
        
        # Products (class 1) - colored rectangles
        np.random.seed(idx)
        for _ in range(np.random.randint(8, 15)):
            x = np.random.randint(50, self.img_size - 100)
            y = np.random.randint(50, self.img_size - 100)
            w, h = np.random.randint(40, 80), np.random.randint(60, 120)
            
            # Random product color
            color = np.random.rand(3)
            image[y:y+h, x:x+w] = color
            mask[y:y+h, x:x+w] = 1
        
        # Empty space (class 2) - gaps between products
        for _ in range(np.random.randint(3, 6)):
            x = np.random.randint(50, self.img_size - 100)
            y = np.random.randint(50, self.img_size - 50)
            w = np.random.randint(30, 60)
            
            image[y:y+40, x:x+w] = [0.8, 0.8, 0.75]  # Light shelf color
            mask[y:y+40, x:x+w] = 2
        
        # Price tags (class 3) - small yellow rectangles
        for _ in range(np.random.randint(5, 10)):
            x = np.random.randint(50, self.img_size - 50)
            y = np.random.randint(50, self.img_size - 30)
            
            image[y:y+20, x:x+30] = [1.0, 1.0, 0.3]  # Yellow tags
            mask[y:y+20, x:x+30] = 3
        
        # Convert to torch tensors
        image = torch.from_numpy(image).permute(2, 0, 1)  # [3, H, W]
        mask = torch.from_numpy(mask).long()  # [H, W]
        
        return image, mask

# Create datasets
train_dataset = SyntheticShelfDataset(num_samples=500, img_size=512)
val_dataset = SyntheticShelfDataset(num_samples=100, img_size=512)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

# Visualize sample
sample_img, sample_mask = train_dataset[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(sample_img.permute(1, 2, 0))
axes[0].set_title('Sample Shelf Image', fontsize=12)
axes[0].axis('off')

axes[1].imshow(sample_mask, cmap='tab10', vmin=0, vmax=4)
axes[1].set_title('Ground Truth Mask\n(0=bg, 1=products, 2=empty, 3=tags, 4=edges)', fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.savefig('img/ch05-sample-data.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

## 3 · U-Net Architecture Implementation

Build U-Net with:
- **Encoder**: 4 downsampling blocks (512→256→128→64→32 spatial resolution)
- **Bottleneck**: 1024 channels at 32×32
- **Decoder**: 4 upsampling blocks with skip connections
- **Output**: 5-class segmentation map

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=5):
        super(UNet, self).__init__()
        
        # Encoder (downsampling path)
        self.enc1 = self.conv_block(in_channels, 64)
        self.enc2 = self.conv_block(64, 128)
        self.enc3 = self.conv_block(128, 256)
        self.enc4 = self.conv_block(256, 512)
        
        # Bottleneck
        self.bottleneck = self.conv_block(512, 1024)
        
        # Decoder (upsampling path)
        self.upconv4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = self.conv_block(1024, 512)  # 512 + 512 from skip
        
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self.conv_block(512, 256)   # 256 + 256 from skip
        
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self.conv_block(256, 128)   # 128 + 128 from skip
        
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self.conv_block(128, 64)    # 64 + 64 from skip
        
        # Final output
        self.out = nn.Conv2d(64, num_classes, 1)
    
    def conv_block(self, in_ch, out_ch):
        """Two 3×3 convs with ReLU (U-Net pattern)"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder with skip connections saved
        enc1 = self.enc1(x)                          # 512×512×64
        enc2 = self.enc2(F.max_pool2d(enc1, 2))      # 256×256×128
        enc3 = self.enc3(F.max_pool2d(enc2, 2))      # 128×128×256
        enc4 = self.enc4(F.max_pool2d(enc3, 2))      # 64×64×512
        
        # Bottleneck
        bottleneck = self.bottleneck(F.max_pool2d(enc4, 2))  # 32×32×1024
        
        # Decoder with skip connections concatenated
        dec4 = self.upconv4(bottleneck)              # 64×64×512
        dec4 = torch.cat([dec4, enc4], dim=1)        # 64×64×1024
        dec4 = self.dec4(dec4)                       # 64×64×512
        
        dec3 = self.upconv3(dec4)                    # 128×128×256
        dec3 = torch.cat([dec3, enc3], dim=1)        # 128×128×512
        dec3 = self.dec3(dec3)                       # 128×128×256
        
        dec2 = self.upconv2(dec3)                    # 256×256×128
        dec2 = torch.cat([dec2, enc2], dim=1)        # 256×256×256
        dec2 = self.dec2(dec2)                       # 256×256×128
        
        dec1 = self.upconv1(dec2)                    # 512×512×64
        dec1 = torch.cat([dec1, enc1], dim=1)        # 512×512×128
        dec1 = self.dec1(dec1)                       # 512×512×64
        
        return self.out(dec1)  # 512×512×num_classes

# Create model
model = UNet(in_channels=3, num_classes=5).to(device)
print(f"\nModel created with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

# Test forward pass
test_input = torch.randn(1, 3, 512, 512).to(device)
test_output = model(test_input)
print(f"Input shape: {test_input.shape} → Output shape: {test_output.shape}")

## 4 · Training Loop with IoU Metric

Train U-Net with:
- **Loss**: Cross-entropy (pixel-wise)
- **Optimizer**: Adam (lr=1e-4)
- **Metric**: Mean IoU (Intersection over Union)

In [ ]:
def calculate_iou(pred_mask, true_mask, num_classes):
    """
    Calculate mean IoU (Intersection over Union) across all classes.
    
    Args:
        pred_mask: [B, H, W] predicted class indices
        true_mask: [B, H, W] ground truth class indices
        num_classes: number of classes
    
    Returns:
        miou: mean IoU (average across all classes)
        class_ious: list of per-class IoU values
    """
    pred_mask = pred_mask.cpu().numpy()
    true_mask = true_mask.cpu().numpy()
    
    ious = []
    for cls in range(num_classes):
        pred_cls = (pred_mask == cls)
        true_cls = (true_mask == cls)
        
        intersection = np.logical_and(pred_cls, true_cls).sum()
        union = np.logical_or(pred_cls, true_cls).sum()
        
        if union == 0:
            iou = float('nan')  # Class not present
        else:
            iou = intersection / union
        
        ious.append(iou)
    
    miou = np.nanmean(ious)
    return miou, ious

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 20
train_losses = []
val_mious = []

print("\nStarting training...\n")

for epoch in range(num_epochs):
    # Training phase
    model.train()
    epoch_loss = 0
    
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        
        outputs = model(images)  # [B, 5, H, W]
        loss = criterion(outputs, masks)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Validation phase
    model.eval()
    val_miou_total = 0
    
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            
            outputs = model(images)
            pred_masks = outputs.argmax(dim=1)  # [B, H, W]
            
            miou, _ = calculate_iou(pred_masks, masks, num_classes=5)
            val_miou_total += miou
    
    avg_miou = val_miou_total / len(val_loader)
    val_mious.append(avg_miou)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}, Val mIoU: {avg_miou:.4f}")

print(f"\n✅ Final validation mIoU: {val_mious[-1]:.4f}")
print(f"Target for constraint #2: IoU ≥ 0.70 (currently {val_mious[-1]:.4f})")

## 5 · Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
axes[0].plot(range(1, num_epochs+1), train_losses, color='#e74c3c', linewidth=2, marker='o')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[0].set_title('Training Loss (U-Net)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Validation mIoU
axes[1].plot(range(1, num_epochs+1), val_mious, color='#3498db', linewidth=2, marker='s')
axes[1].axhline(y=0.70, color='#2ecc71', linestyle='--', linewidth=2, label='Target (IoU ≥ 0.70)')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Mean IoU', fontsize=12)
axes[1].set_title('Validation mIoU (Constraint #2)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('img/ch05-training-curves.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

## 6 · Visualize Segmentation Results

In [ ]:
# Get sample predictions
model.eval()
sample_images, sample_masks = next(iter(val_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    sample_outputs = model(sample_images)
    sample_preds = sample_outputs.argmax(dim=1)  # [B, H, W]

# Visualize first 3 samples
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

class_names = ['Background', 'Products', 'Empty Space', 'Price Tags', 'Shelf Edges']
colors = ['#2c3e50', '#e74c3c', '#f39c12', '#f1c40f', '#8b4513']

for i in range(3):
    # Original image
    img = sample_images[i].cpu().permute(1, 2, 0).numpy()
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Input Image', fontsize=12)
    axes[i, 0].axis('off')
    
    # Ground truth mask
    gt_mask = sample_masks[i].numpy()
    axes[i, 1].imshow(gt_mask, cmap='tab10', vmin=0, vmax=4)
    axes[i, 1].set_title('Ground Truth Mask', fontsize=12)
    axes[i, 1].axis('off')
    
    # Predicted mask
    pred_mask = sample_preds[i].cpu().numpy()
    axes[i, 2].imshow(pred_mask, cmap='tab10', vmin=0, vmax=4)
    axes[i, 2].set_title('Predicted Mask (U-Net)', fontsize=12)
    axes[i, 2].axis('off')
    
    # Calculate per-sample IoU
    miou, class_ious = calculate_iou(sample_preds[i:i+1], sample_masks[i:i+1], num_classes=5)
    axes[i, 2].text(0.5, -0.1, f'mIoU: {miou:.3f}', 
                    transform=axes[i, 2].transAxes, 
                    ha='center', fontsize=10, color='#2ecc71')

plt.tight_layout()
plt.savefig('img/ch05-segmentation-results.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("\nClass Legend:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

## 7 · Per-Class IoU Analysis

In [ ]:
# Calculate per-class IoU on full validation set
model.eval()
all_class_ious = [[] for _ in range(5)]

with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        pred_masks = outputs.argmax(dim=1)
        
        _, class_ious = calculate_iou(pred_masks, masks, num_classes=5)
        
        for cls in range(5):
            if not np.isnan(class_ious[cls]):
                all_class_ious[cls].append(class_ious[cls])

# Average per-class IoU
avg_class_ious = [np.mean(ious) if ious else 0 for ious in all_class_ious]

# Plot bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(class_names))
bars = ax.bar(x_pos, avg_class_ious, color=['#2c3e50', '#e74c3c', '#f39c12', '#f1c40f', '#8b4513'], 
              edgecolor='white', linewidth=1.5)

# Add value labels on bars
for i, (bar, iou) in enumerate(zip(bars, avg_class_ious)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{iou:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('IoU Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class IoU Performance (U-Net on Retail Shelves)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(class_names, rotation=15, ha='right')
ax.axhline(y=0.70, color='#2ecc71', linestyle='--', linewidth=2, label='Target IoU ≥ 0.70')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('img/ch05-per-class-iou.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("\nPer-Class IoU Summary:")
for name, iou in zip(class_names, avg_class_ious):
    status = "✅" if iou >= 0.70 else "⚡"
    print(f"  {status} {name}: {iou:.4f}")

overall_miou = np.mean(avg_class_ious)
print(f"\n📊 Overall mIoU: {overall_miou:.4f}")
print(f"🎯 Constraint #2 status: {'✅ Met' if overall_miou >= 0.70 else '⚡ In Progress'} (target ≥ 0.70)")

## 8 · What Can Go Wrong: Class Imbalance

Demonstrate class imbalance problem: 80% background pixels dominate the loss.

In [ ]:
# Calculate class distribution in validation set
class_pixel_counts = np.zeros(5)

for images, masks in val_loader:
    for cls in range(5):
        class_pixel_counts[cls] += (masks == cls).sum().item()

total_pixels = class_pixel_counts.sum()
class_percentages = (class_pixel_counts / total_pixels) * 100

# Visualize class distribution
fig, ax = plt.subplots(figsize=(10, 6))
colors_dist = ['#2c3e50', '#e74c3c', '#f39c12', '#f1c40f', '#8b4513']
bars = ax.bar(class_names, class_percentages, color=colors_dist, edgecolor='white', linewidth=1.5)

for bar, pct in zip(bars, class_percentages):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Percentage of Pixels (%)', fontsize=12, fontweight='bold')
ax.set_title('Class Distribution in Validation Set', fontsize=14, fontweight='bold')
ax.set_xticklabels(class_names, rotation=15, ha='right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('img/ch05-class-imbalance.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

print("⚠️ Class Imbalance Problem:")
print(f"  Background dominates with {class_percentages[0]:.1f}% of pixels")
print(f"  Minority classes (price tags) only {class_percentages[3]:.1f}% of pixels")
print("\n💡 Solution: Use weighted loss or focal loss to balance class contributions")

## 9 · Exercises

**Exercise 1:** Implement weighted cross-entropy loss to handle class imbalance.  
**Exercise 2:** Reduce output stride from 16 to 8 — does it improve boundary quality?  
**Exercise 3:** Replace U-Net encoder with ResNet-50 — compare accuracy vs speed.

In [ ]:
# Exercise 1: Weighted Cross-Entropy Loss
# TODO: Calculate inverse frequency weights for each class
# weights = torch.tensor([1.0 / (count + 1e-5) for count in class_pixel_counts])
# weighted_criterion = nn.CrossEntropyLoss(weight=weights.to(device))

# Exercise 2: Atrous Convolutions for Higher Resolution
# TODO: Modify U-Net to use dilated convolutions in bottleneck
# This maintains spatial resolution without increasing computation

# Exercise 3: ResNet-50 Encoder
# TODO: Replace custom encoder with pretrained ResNet-50
# from torchvision.models import resnet50
# encoder = resnet50(pretrained=True)
# Use layers 1-4 as encoder blocks

print("💡 Exercises for further exploration:")
print("  1. Implement weighted loss to address class imbalance")
print("  2. Experiment with output stride (8 vs 16 vs 32)")
print("  3. Swap encoder for ResNet-50 (transfer learning)")